In [2]:
import sqlite3

In [3]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [4]:
def get_ticket_price(city):
    print(f"데이터베이스 도구 호출됨: {city}의 가격을 조회합니다", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"{city}로 가는 티켓 가격은 ${result[0]}입니다" if result else "이 도시에 대한 가격 정보가 없습니다"

In [5]:
get_ticket_price("London")

데이터베이스 도구 호출됨: London의 가격을 조회합니다


'이 도시에 대한 가격 정보가 없습니다'

In [6]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [7]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [9]:
get_ticket_price("tokyo")

데이터베이스 도구 호출됨: tokyo의 가격을 조회합니다


'tokyo로 가는 티켓 가격은 $1420.0입니다'

In [10]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [11]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API 키가 존재하며 {google_api_key[:8]}로 시작합니다")
else:
    print("Google API 키가 설정되어 있지 않습니다")


Google API 키가 존재하며 AQ.Ab8RN로 시작합니다


In [12]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = "gemini-3.1-flash-lite"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [13]:
system_message = """
당신은 FlightAI라는 항공사의 도움이 되는 어시스턴트입니다.
짧고 정중한 답변을 하며, 1문장을 넘기지 마세요.
항상 정확하게 답하세요. 답을 모른다면, 모른다고 말하세요.
"""

In [14]:
price_function = {
    "name" : "get_ticket_price",
    "description" : "목적지 도시로 가는 왕복 티켓의 가격을 가져옵니다.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type" : "string",
                "description" : "고객이 여행하고자 하는 도시",
            },
        },
        "required" : ["destination_city"],
        "additionalProperties" : False
    }
}

In [15]:
tools = [{"type" : "function", "function": price_function}]

In [16]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            price_details = get_ticket_price(city)
            responses.append({
                "role" : "tool",
                "content" : price_details,
                "tool_call_id" : tool_call.id
            })
    return responses

In [19]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responsees = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responsees)
        response = gemini.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


데이터베이스 도구 호출됨: Tokyo의 가격을 조회합니다
데이터베이스 도구 호출됨: Tokyo의 가격을 조회합니다
데이터베이스 도구 호출됨: London의 가격을 조회합니다
